# Spectral Analysis — Consolidated Results (L-SML v2, Oriented)

**Step 120 notebook.** Re-runs all Consolidated Results fusion using the oriented
L-SML v2 pipeline derived from Jaffé-Fetaya-Nadler (2016):

1. Pre-orient each feature with consensus `FEATURE_SIGNS` (Step 110 cross-dataset analysis)
2. Binarize at empirical median via `binarize_classifiers`
3. Fuse binary classifiers with `lsml_fuse` (Algorithm 2)

**Difference from Step 107 notebook (`sml_unsupervised`):**

| Aspect | Step 107 | v2 (this notebook) |
|---|---|---|
| Sign orientation | Eigenvector majority vote (assumption iii) | **FEATURE_SIGNS — Step 110 consensus** |
| Binarization | Median split (no pre-orientation) | **Median split after orientation** |
| Labels used for | AUROC only | AUROC only |

**Re-uses cached feature pkls from `consolidated_results/` on Drive — no GPU needed.**

**Outputs (all in `consolidated_results/`):**
- `lsml_v2_math500_res.pkl`, `lsml_v2_gsm8k_res.pkl`, `lsml_v2_gpqa_res.pkl`
- `lsml_v2_rag_res.pkl`, `lsml_v2_qa_res.pkl`
- `lsml_v2_results_all.pkl` — combined
- `lsml_v2_summary.csv` — per-cell AUROC table

**Step 100 `results_all.pkl` and Step 107 `lsml_results_all.pkl` are untouched.**

## Section 1 — Setup

In [3]:
import os, sys, shutil

REPO_DIR = '/content/hallucination_detection'

if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b master https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull -q')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.system('pip install -q "transformers>=4.40" accelerate datasets scipy scikit-learn')

import numpy as np
import pickle

from spectral_utils import (
    FEAT_NAMES, boot_auc, binarize_classifiers, lsml_fuse,
)

print('spectral_utils imported OK')
print(f'FEAT_NAMES ({len(FEAT_NAMES)}): {FEAT_NAMES}')

spectral_utils imported OK
FEAT_NAMES (16): ['epr', 'trace_length', 'spectral_entropy', 'low_band_power', 'high_band_power', 'hl_ratio', 'dominant_freq', 'spectral_centroid', 'stft_max_high_power', 'stft_spectral_entropy', 'rpdi', 'sw_var_peak', 'pe_mean', 'hurst_exponent', 'cusum_max', 'cusum_shift_idx']


In [4]:
from google.colab import drive
drive.mount('/content/drive')

BASE     = '/content/drive/MyDrive'
HALL_DIR = f'{BASE}/hallucination_detection'
OUT_DIR  = f'{HALL_DIR}/consolidated_results'
os.makedirs(OUT_DIR, exist_ok=True)

print(f'OUT_DIR = {OUT_DIR}  (exists: {os.path.exists(OUT_DIR)})')

# FEATURE_SIGNS: Step 110 consensus across 29 cells
# +1 = higher raw value -> more likely correct
# -1 = higher raw value -> hallucination / wrong answer
FEATURE_SIGNS = {
    'epr': -1, 'trace_length': 1, 'spectral_entropy': -1,
    'low_band_power': -1, 'high_band_power': -1, 'hl_ratio': -1,
    'dominant_freq': -1, 'spectral_centroid': -1,
    'stft_max_high_power': -1, 'stft_spectral_entropy': -1,
    'rpdi': -1, 'sw_var_peak': -1,
    'pe_mean': -1, 'hurst_exponent': 1,
    'cusum_max': -1, 'cusum_shift_idx': 1,
}

# Step 121-123 ablation: 5 features with consistently clean individual AUROC signal.
# Derived offline from 29 cached cells; unsupervised at test time.
GOOD_FEATURES = ['epr', 'low_band_power', 'sw_var_peak', 'cusum_max', 'spectral_entropy']

CACHED_FEAT_PKLS = {
    'math500': os.path.join(OUT_DIR, 'math500_res.pkl'),
    'gsm8k':   os.path.join(OUT_DIR, 'gsm8k_res.pkl'),
    'gpqa':    os.path.join(OUT_DIR, 'gpqa_res.pkl'),
    'rag':     os.path.join(OUT_DIR, 'rag_feats_all.pkl'),
    'qa':      os.path.join(OUT_DIR, 'qa_res.pkl'),
}

for name, path in CACHED_FEAT_PKLS.items():
    exists = os.path.exists(path)
    print(f'  {name}: {"OK" if exists else "MISSING"} -- {path}')
print(f'GOOD_FEATURES ({len(GOOD_FEATURES)}): {GOOD_FEATURES}')

Mounted at /content/drive
OUT_DIR = /content/drive/MyDrive/hallucination_detection/consolidated_results  (exists: True)
  math500: OK -- /content/drive/MyDrive/hallucination_detection/consolidated_results/math500_res.pkl
  gsm8k: OK -- /content/drive/MyDrive/hallucination_detection/consolidated_results/gsm8k_res.pkl
  gpqa: OK -- /content/drive/MyDrive/hallucination_detection/consolidated_results/gpqa_res.pkl
  rag: OK -- /content/drive/MyDrive/hallucination_detection/consolidated_results/rag_feats_all.pkl
  qa: OK -- /content/drive/MyDrive/hallucination_detection/consolidated_results/qa_res.pkl
GOOD_FEATURES (5): ['epr', 'low_band_power', 'sw_var_peak', 'cusum_max', 'spectral_entropy']


In [5]:
def load_cached_feats(pkl_path):
    """
    Load cached (feats_dict, labels) pairs from a pkl produced by the
    Consolidated Results notebook.  Handles two on-disk formats:
      1. {'feats': {key: (fd, lbl)}, 'results': ...}
      2. {key: (fd, lbl)} directly (e.g. rag_feats_all.pkl)
    Returns dict {key: (fd, lbl)} or None if the file is missing.
    """
    if not os.path.exists(pkl_path):
        return None
    with open(pkl_path, 'rb') as f:
        obj = pickle.load(f)
    if isinstance(obj, dict) and 'feats' in obj:
        return obj['feats']
    return obj


def run_lsml_v2(fd, lbl, key_str, verbose=True):
    """
    Oriented L-SML v2 pipeline with feature selection (Step 121-123).

    Steps:
      1. binarize_classifiers(fd, FEATURE_SIGNS) -- orients then median-splits all 16
      2. filter to GOOD_FEATURES                 -- keep 5 informative classifiers
      3. lsml_fuse(*binary_filt.values())         -- Algorithm 2 (Jaffe et al. 2016)
      4. boot_auc(lbl, fused)                    -- AUROC, labels for eval only

    Returns a result dict or None if the cell is degenerate (one class).
    """
    lbl = np.asarray(lbl, dtype=int)
    if len(set(lbl.tolist())) < 2:
        if verbose:
            print(f'  [{key_str}] only one class -- skip')
        return None
    n_pos = int(lbl.sum())
    n_neg = int(len(lbl) - n_pos)

    # Individual feature AUROCs oriented by FEATURE_SIGNS (no label leakage)
    ind_aucs = {}
    for fn in FEAT_NAMES:
        try:
            sign = FEATURE_SIGNS.get(fn, +1)
            a, *_ = boot_auc(lbl, np.array(fd[fn], dtype=float) * sign)
            ind_aucs[fn] = float(a) if not (a != a) else float('nan')  # nan check
        except Exception:
            ind_aucs[fn] = float('nan')

    try:
        binary = binarize_classifiers(fd, FEATURE_SIGNS)
        binary_filt = {fn: binary[fn] for fn in GOOD_FEATURES if fn in binary}
        fused, meta = lsml_fuse(*binary_filt.values())
    except Exception as e:
        if verbose:
            print(f'  [{key_str}] L-SML v2 error: {e}')
        return None

    # Safety: take max(AUC, 1-AUC) in case the fused sign is flipped
    p_auc, p_lo, p_hi = boot_auc(lbl,  fused)
    n_auc, n_lo, n_hi = boot_auc(lbl, -fused)
    if p_auc >= n_auc:
        v2_auc, v2_lo, v2_hi = p_auc, p_lo, p_hi
    else:
        v2_auc, v2_lo, v2_hi = n_auc, n_lo, n_hi

    if verbose:
        print(f'  [{key_str}] N={len(lbl)} (+{n_pos}/-{n_neg}) | '
              f'v2 L-SML={v2_auc:.3f} [{v2_lo:.3f},{v2_hi:.3f}] K={meta["K"]}')

    return {
        'n': len(lbl), 'n_pos': n_pos, 'n_neg': n_neg,
        'ind_aucs': ind_aucs,
        'v2_auc': v2_auc, 'v2_lo': v2_lo, 'v2_hi': v2_hi,
        'K': meta['K'],
        'c': meta['c'].tolist(),
        'group_weights': [(idx.tolist(), w.tolist()) for idx, w in meta['group_weights']],
        'cross_weights': meta['cross_weights'].tolist(),
        'residual': float(meta['residual']),
        'method': meta['method'],
    }


print('Helpers defined.')

Helpers defined.


## Section 2 — MATH-500

In [6]:
V2_PATH = os.path.join(OUT_DIR, 'lsml_v2_math500_res.pkl')
FORCE = True  # recompute with 5-feature GOOD_FEATURES filter

MATH500_FEATS = load_cached_feats(CACHED_FEAT_PKLS['math500'])
if MATH500_FEATS is None:
    print('ERROR: math500 pkl not found. Run Spectral_Analysis_Consolidated_Results.ipynb first.')
    MATH500_V2 = {}
else:
    if not FORCE and os.path.exists(V2_PATH):
        with open(V2_PATH, 'rb') as f:
            MATH500_V2 = pickle.load(f)
        remaining = [k for k in MATH500_FEATS if k not in MATH500_V2]
        print(f'Loaded {len(MATH500_V2)} cached MATH-500 keys; {len(remaining)} remaining')
    else:
        MATH500_V2 = {}
        remaining = list(MATH500_FEATS.keys())
        print(f'Running L-SML v2 on MATH-500 ({len(remaining)} keys)...')

    for key in remaining:
        fd, lbl = MATH500_FEATS[key]
        MATH500_V2[key] = run_lsml_v2(fd, lbl, f'MATH-500/{key}')
        with open(V2_PATH, 'wb') as f:
            pickle.dump(MATH500_V2, f)

    if remaining:
        print(f'Saved {V2_PATH} ({len(MATH500_V2)} total keys)')
    else:
        print(f'All {len(MATH500_V2)} MATH-500 keys already computed')

Running L-SML v2 on MATH-500 (4 keys)...


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [MATH-500/Qwen2.5-Math-1.5B-Instruct_T1.0] N=300 (+133/-167) | v2 L-SML=0.832 [0.787,0.873] K=2


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [MATH-500/Qwen-Math-7B_T1.0] N=300 (+84/-216) | v2 L-SML=0.882 [0.840,0.920] K=2


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [MATH-500/deepseek-math-7b-instruct_T1.0] N=300 (+59/-241) | v2 L-SML=0.660 [0.582,0.728] K=3


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [MATH-500/DeepSeek-R1-Distill-Llama-8B_T1.0] N=300 (+123/-177) | v2 L-SML=0.816 [0.767,0.864] K=2
Saved /content/drive/MyDrive/hallucination_detection/consolidated_results/lsml_v2_math500_res.pkl (4 total keys)


## Section 3 — GSM8K

In [7]:
V2_PATH = os.path.join(OUT_DIR, 'lsml_v2_gsm8k_res.pkl')
FORCE = True  # recompute with 5-feature GOOD_FEATURES filter

GSM8K_FEATS = load_cached_feats(CACHED_FEAT_PKLS['gsm8k'])
if GSM8K_FEATS is None:
    print('ERROR: gsm8k pkl not found. Run Spectral_Analysis_Consolidated_Results.ipynb first.')
    GSM8K_V2 = {}
else:
    if not FORCE and os.path.exists(V2_PATH):
        with open(V2_PATH, 'rb') as f:
            GSM8K_V2 = pickle.load(f)
        remaining = [k for k in GSM8K_FEATS if k not in GSM8K_V2]
        print(f'Loaded {len(GSM8K_V2)} cached GSM8K keys; {len(remaining)} remaining')
    else:
        GSM8K_V2 = {}
        remaining = list(GSM8K_FEATS.keys())
        print(f'Running L-SML v2 on GSM8K ({len(remaining)} keys)...')

    for key in remaining:
        fd, lbl = GSM8K_FEATS[key]
        GSM8K_V2[key] = run_lsml_v2(fd, lbl, f'GSM8K/{key}')
        with open(V2_PATH, 'wb') as f:
            pickle.dump(GSM8K_V2, f)

    if remaining:
        print(f'Saved {V2_PATH} ({len(GSM8K_V2)} total keys)')
    else:
        print(f'All {len(GSM8K_V2)} GSM8K keys already computed')

Running L-SML v2 on GSM8K (1 keys)...


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [GSM8K/Llama-8B_T1.0] N=1319 (+1043/-276) | v2 L-SML=0.707 [0.672,0.740] K=2
Saved /content/drive/MyDrive/hallucination_detection/consolidated_results/lsml_v2_gsm8k_res.pkl (1 total keys)


## Section 4 — GPQA

In [8]:
V2_PATH = os.path.join(OUT_DIR, 'lsml_v2_gpqa_res.pkl')
FORCE = True  # recompute with 5-feature GOOD_FEATURES filter

GPQA_FEATS = load_cached_feats(CACHED_FEAT_PKLS['gpqa'])
if GPQA_FEATS is None:
    print('ERROR: gpqa pkl not found. Run Spectral_Analysis_Consolidated_Results.ipynb first.')
    GPQA_V2 = {}
else:
    if not FORCE and os.path.exists(V2_PATH):
        with open(V2_PATH, 'rb') as f:
            GPQA_V2 = pickle.load(f)
        remaining = [k for k in GPQA_FEATS if k not in GPQA_V2]
        print(f'Loaded {len(GPQA_V2)} cached GPQA keys; {len(remaining)} remaining')
    else:
        GPQA_V2 = {}
        remaining = list(GPQA_FEATS.keys())
        print(f'Running L-SML v2 on GPQA ({len(remaining)} keys)...')

    for key in remaining:
        fd, lbl = GPQA_FEATS[key]
        GPQA_V2[key] = run_lsml_v2(fd, lbl, f'GPQA/{key}')
        with open(V2_PATH, 'wb') as f:
            pickle.dump(GPQA_V2, f)

    if remaining:
        print(f'Saved {V2_PATH} ({len(GPQA_V2)} total keys)')
    else:
        print(f'All {len(GPQA_V2)} GPQA keys already computed')

Running L-SML v2 on GPQA (5 keys)...


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [GPQA/Llama-8B_T1.0] N=198 (+53/-145) | v2 L-SML=0.511 [0.419,0.602] K=4


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [GPQA/Qwen-7B_T1.0] N=198 (+60/-138) | v2 L-SML=0.536 [0.448,0.622] K=3


...

  [GPQA/DeepSeek-R1-Distill-Llama-8B_T1.0] N=198 (+48/-150) | v2 L-SML=0.544 [0.447,0.643] K=3


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [GPQA/Mistral-7B_T1.0] N=198 (+50/-148) | v2 L-SML=0.555 [0.468,0.640] K=2


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [GPQA/Qwen2.5-72B-Instruct-AWQ_T1.0] N=198 (+80/-118) | v2 L-SML=0.514 [0.444,0.586] K=2
Saved /content/drive/MyDrive/hallucination_detection/consolidated_results/lsml_v2_gpqa_res.pkl (5 total keys)


## Section 5 — RAG (L-CiteEval)

In [9]:
V2_PATH = os.path.join(OUT_DIR, 'lsml_v2_rag_res.pkl')
FORCE = True  # recompute with 5-feature GOOD_FEATURES filter

RAG_FEATS = load_cached_feats(CACHED_FEAT_PKLS['rag'])
if RAG_FEATS is None:
    print('ERROR: rag pkl not found. Run Spectral_Analysis_Consolidated_Results.ipynb first.')
    RAG_V2 = {}
else:
    if not FORCE and os.path.exists(V2_PATH):
        with open(V2_PATH, 'rb') as f:
            RAG_V2 = pickle.load(f)
        remaining = [k for k in RAG_FEATS if k not in RAG_V2]
        print(f'Loaded {len(RAG_V2)} cached RAG keys; {len(remaining)} remaining')
    else:
        RAG_V2 = {}
        remaining = list(RAG_FEATS.keys())
        print(f'Running L-SML v2 on RAG ({len(remaining)} keys)...')

    for key in remaining:
        fd, lbl = RAG_FEATS[key]
        RAG_V2[key] = run_lsml_v2(fd, lbl, f'RAG/{key}')
        with open(V2_PATH, 'wb') as f:
            pickle.dump(RAG_V2, f)

    if remaining:
        print(f'Saved {V2_PATH} ({len(RAG_V2)} total keys)')
    else:
        print(f'All {len(RAG_V2)} RAG keys already computed')

Running L-SML v2 on RAG (16 keys)...


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [RAG/Qwen-7B/hotpotqa] N=169 (+15/-154) | v2 L-SML=0.548 [0.419,0.695] K=2


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [RAG/Qwen-7B/natural-questions] N=103 (+11/-92) | v2 L-SML=0.571 [0.405,0.727] K=2


...

  [RAG/Llama-8B/natural-questions] N=121 (+8/-113) | v2 L-SML=0.580 [0.334,0.811] K=5


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [RAG/Llama-8B/2wikimultihopqa] N=99 (+25/-74) | v2 L-SML=0.556 [0.436,0.674] K=2


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [RAG/Llama-8B/narrativeqa] N=459 (+67/-392) | v2 L-SML=0.569 [0.484,0.645] K=4
Saved /content/drive/MyDrive/hallucination_detection/consolidated_results/lsml_v2_rag_res.pkl (16 total keys)


## Section 6 — Factual QA

In [10]:
V2_PATH = os.path.join(OUT_DIR, 'lsml_v2_qa_res.pkl')
FORCE = True  # recompute with 5-feature GOOD_FEATURES filter

QA_FEATS = load_cached_feats(CACHED_FEAT_PKLS['qa'])
if QA_FEATS is None:
    print('ERROR: qa pkl not found. Run Spectral_Analysis_Consolidated_Results.ipynb first.')
    QA_V2 = {}
else:
    if not FORCE and os.path.exists(V2_PATH):
        with open(V2_PATH, 'rb') as f:
            QA_V2 = pickle.load(f)
        remaining = [k for k in QA_FEATS if k not in QA_V2]
        print(f'Loaded {len(QA_V2)} cached QA keys; {len(remaining)} remaining')
    else:
        QA_V2 = {}
        remaining = list(QA_FEATS.keys())
        print(f'Running L-SML v2 on QA ({len(remaining)} keys)...')

    for key in remaining:
        fd, lbl = QA_FEATS[key]
        QA_V2[key] = run_lsml_v2(fd, lbl, f'QA/{key}')
        with open(V2_PATH, 'wb') as f:
            pickle.dump(QA_V2, f)

    if remaining:
        print(f'Saved {V2_PATH} ({len(QA_V2)} total keys)')
    else:
        print(f'All {len(QA_V2)} QA keys already computed')

Running L-SML v2 on QA (4 keys)...


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [QA/spectral_phase9_cache_trivia_qa_traces_T1.0] N=52 (+2/-50) | v2 L-SML=0.690 [0.571,0.794] K=3
  [QA/spectral_phase9_cache_webq_traces_T1.0] only one class -- skip


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [QA/spectral_phase9_cache_trivia_qa_cot_traces_T1.0] N=285 (+79/-206) | v2 L-SML=0.610 [0.540,0.684] K=3


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  _, diffusion_map = eigsh(


  [QA/spectral_phase9_cache_webq_cot_traces_T1.0] N=290 (+33/-257) | v2 L-SML=0.587 [0.502,0.675] K=3
Saved /content/drive/MyDrive/hallucination_detection/consolidated_results/lsml_v2_qa_res.pkl (4 total keys)


## Section 7 — Summary Table

In [11]:
# (1) Save combined pkl
all_pkl = os.path.join(OUT_DIR, 'lsml_v2_results_all.pkl')
with open(all_pkl, 'wb') as f:
    pickle.dump({
        'math500': MATH500_V2, 'gsm8k': GSM8K_V2, 'gpqa': GPQA_V2,
        'rag': RAG_V2, 'qa': QA_V2,
    }, f)
print(f'Saved combined pkl -> {all_pkl}')

# (2) Build summary dataframe
import pandas as pd

domain_map = [
    ('MATH500', MATH500_V2), ('GSM8K', GSM8K_V2), ('GPQA', GPQA_V2),
    ('RAG', RAG_V2), ('QA', QA_V2),
]
all_v2 = {}
for domain, dres in domain_map:
    for k, v in dres.items():
        if v:
            all_v2[f'{domain}/{k}'] = v

rows = []
for full_key, res in sorted(all_v2.items(), key=lambda x: -x[1]['v2_auc']):
    rows.append({
        'domain_model': full_key,
        'v2_auc':       round(res['v2_auc'], 4),
        'v2_lo':        round(res['v2_lo'], 4),
        'v2_hi':        round(res['v2_hi'], 4),
        'v2_ci':        f"[{res['v2_lo']:.3f},{res['v2_hi']:.3f}]",
        'K':            res['K'],
        'n':            res['n'],
        'n_pos':        res['n_pos'],
        'n_neg':        res['n_neg'],
    })

summary_df = pd.DataFrame(rows)
csv_path = os.path.join(OUT_DIR, 'lsml_v2_summary.csv')
summary_df.to_csv(csv_path, index=False)
print(f'Saved summary CSV -> {csv_path}')

# (3) Print results table
print()
print('=' * 90)
print(f'{"Domain/Model":<45} {"v2 L-SML AUROC":>16} {"95% CI":>20} {"K":>4} {"N":>6}')
print('=' * 90)
for _, row in summary_df.iterrows():
    print(f'{row["domain_model"]:<45} {100*row["v2_auc"]:>15.1f}% {row["v2_ci"]:>20} {row["K"]:>4} {row["n"]:>6}')
print('=' * 90)
print(f'Total cells with results: {len(summary_df)}')
valid = summary_df[summary_df['v2_auc'] >= 0.5]
print(f'Cells beating chance (AUROC >= 0.50): {len(valid)}/{len(summary_df)}')

Saved combined pkl -> /content/drive/MyDrive/hallucination_detection/consolidated_results/lsml_v2_results_all.pkl
Saved summary CSV -> /content/drive/MyDrive/hallucination_detection/consolidated_results/lsml_v2_summary.csv

Domain/Model                                    v2 L-SML AUROC               95% CI    K      N
MATH500/Qwen-Math-7B_T1.0                                88.2%        [0.840,0.920]    2    300
MATH500/Qwen2.5-Math-1.5B-Instruct_T1.0                  83.2%        [0.787,0.873]    2    300
MATH500/DeepSeek-R1-Distill-Llama-8B_T1.0                81.6%        [0.767,0.864]    2    300
RAG/Llama-8B/hotpotqa                                    74.3%        [0.650,0.830]    2     93
GSM8K/Llama-8B_T1.0                                      70.7%        [0.672,0.740]    2   1319
RAG/Qwen-72B/hotpotqa                                    69.2%        [0.597,0.778]    3    169
QA/spectral_phase9_cache_trivia_qa_traces_T1.0            69.0%        [0.571,0.794]    3     52
MATH500

## Done

**Saved to Drive `consolidated_results/`:**
- `lsml_v2_math500_res.pkl`
- `lsml_v2_gsm8k_res.pkl`
- `lsml_v2_gpqa_res.pkl`
- `lsml_v2_rag_res.pkl`
- `lsml_v2_qa_res.pkl`
- `lsml_v2_results_all.pkl` — combined; keys: `math500`, `gsm8k`, `gpqa`, `rag`, `qa`
- `lsml_v2_summary.csv` — per-cell AUROC table (sorted by AUROC desc)

**Per-key result dict fields:**
- `v2_auc`, `v2_lo`, `v2_hi` — AUROC with 95% bootstrap CI
- `K` — number of dependent groups detected by L-SML
- `c` — group assignment array
- `group_weights` — per-group SML weights `[(idx_list, weights_list)]`
- `cross_weights` — cross-group SML weights
- `residual` — Algorithm 1 residual at best K
- `ind_aucs` — individual feature AUROCs (oriented by FEATURE_SIGNS)
- `n`, `n_pos`, `n_neg` — sample counts